In [ ]:
import os, time
from pathlib import Path
import sys
import pandas as pd
import numpy as np

project_root = Path.cwd()
while not (project_root / "src").exists():
    project_root = project_root.parent
sys.path.append(str(project_root))

from src.utilities.project_paths import RAW_DIR, DATA_DIR

pd.set_option('display.float_format', '{:,.2f}'.format)

CENSUS_DIR  = RAW_DIR / 'south_africa' / 'Census2022SampleSTATA'
PARQUET_DIR = DATA_DIR / '01_interim' / 'sa_census_parquet'
PARQUET_DIR.mkdir(parents=True, exist_ok=True)

HOUSEHOLDS_DTA = CENSUS_DIR / 'Census2022Households.dta'
PERSONS_DTA    = CENSUS_DIR / 'Census2022Persons.dta'
GEOGRAPHY_DTA  = CENSUS_DIR / 'Census2022Geography.dta'

# The Persons file is the big one (hundreds of MB). On a shared server we never
# read it in full: we take at most SAMPLE_N rows, once, via a Stata iterator.
SAMPLE_N = 100_000

# Stata value labels -> readable keys for the summary tables.
PROVINCE_LABELS = {1: 'Western Cape', 2: 'Eastern Cape', 3: 'Northern Cape',
                   4: 'Free State', 5: 'KwaZulu-Natal', 6: 'North West',
                   7: 'Gauteng', 8: 'Mpumalanga', 9: 'Limpopo'}
HHPOP_LABELS    = {1: 'Black African', 2: 'Coloured', 3: 'Indian or Asian',
                   4: 'White', 5: 'Other'}
GEOTYPE_LABELS  = {1: 'Urban area', 2: 'Tribal/traditional area', 3: 'Farm area'}

print('Census DTA dir :', CENSUS_DIR)
print('Parquet dir    :', PARQUET_DIR)
print('Source files:')
for p in (HOUSEHOLDS_DTA, PERSONS_DTA, GEOGRAPHY_DTA):
    print(f'  {p.name:28s} {p.stat().st_size / 1024**2:7.1f} MB')
print(f'Persons sample : first {SAMPLE_N:,} rows only (never the whole file)')

---
# Part A — Smart Reading: Column Selection & Dtype Hints

We use the **South Africa Census 2022 sample**, split across three Stata files
that share the household key `QID`:

| File | Grain | Size | Key columns |
|---|---|---|---|
| `Census2022Households.dta` | one row per household | ~62 MB | `DERH_HSIZE`, `DERH_HHPOP`, `H03_TENURE` |
| `Census2022Persons.dta` | one row per person | **~347 MB** | `P04_AGE`, `P02_SEX` |
| `Census2022Geography.dta` | one row per household | ~26 MB | `Province`, `District`, `Geo_type` |

**Two habits to apply:**
1. `columns=[...]` — load only what you need
2. Keep numeric codes numeric, and convert low-cardinality codes to `"category"`
   - `convert_categoricals=False` keeps Stata codes as small integers (so a size of `10`
     stays a number, not the label `"10 +"`)
   - `.astype("category")` then compresses repeating codes. For CSV/Excel, pass
     `dtype={"col": "category"}` directly to `read_csv`/`read_excel`.

## A1. Profile the full main file

In [ ]:
# TODO:
# 1. Load Census2022Households.dta in full (HOUSEHOLDS_DTA)
# 2. Print shape and total memory in MB
#    df.memory_usage(deep=True).sum() / 1024**2

## A2. Column-selective load with dtype conversion

Task: *"Compute mean household size by population group and tenure status."*

Columns needed: `DERH_HHPOP` (population group of head), `H03_TENURE` (tenure),
`DERH_HHSEX` (sex of head), `DERH_HSIZE` (household size), `DERH_HHAGE` (age of head).

Read with `convert_categoricals=False` so `DERH_HSIZE` stays numeric (`10` = "10 or more"),
then convert the low-cardinality demographic codes to `"category"`.

In [ ]:
TASK_COLS = ['DERH_HHPOP', 'H03_TENURE', 'DERH_HHSEX', 'DERH_HSIZE', 'DERH_HHAGE']

# TODO:
# 1. Load Households with columns=TASK_COLS and convert_categoricals=False
#    (so DERH_HSIZE stays a number, not the label "10 +")
# 2. Convert DERH_HHPOP, H03_TENURE and DERH_HHSEX to 'category' with .astype()
#    (For CSV/Excel, pass dtype={...} directly to read_csv/read_excel instead)
# 3. Print shape and memory (MB) before and after the astype conversion
# 4. Print the reduction factor vs the full load from A1

## A3. Grouped analysis from the slim load

In [ ]:
# TODO (uses the slim DataFrame from A2):
# 1. groupby ['DERH_HHPOP', 'H03_TENURE'] (use observed=True with category dtypes)
# 2. Compute: mean of DERH_HSIZE, mean of DERH_HHAGE, count of households
# 3. Sort by mean_hhsize descending, display top 15
# 4. (optional) map DERH_HHPOP codes to labels with HHPOP_LABELS for readability

---
# Part B — Vectorised Operations

**The rule:** reach for `.apply()` only when no vectorised form exists.

| Task | Slow | Fast |
|---|---|---|
| Conditional column | `df.apply(lambda r: ...)` | `np.where(...)` / `np.select(...)` |
| Age groups / bins | `df.apply(classify)` | `pd.cut(...)` |
| String cleaning | `df.col.apply(str.strip)` | `df.col.str.strip()` |
| Arithmetic with guard | `df.apply(lambda r: r.a/r.b if r.b else None)` | `df.a / df.b.where(df.b != 0)` |

## B1. Benchmark: `.apply()` vs `pd.cut` for age grouping

Load `P04_AGE` from the **Persons** file and classify each person into:
`"child"` (<= 14), `"working_age"` (15-64), `"elderly"` (>= 65).

We never read the whole 347 MB Persons file — we pull the first `SAMPLE_N` rows with a
Stata iterator. Measure how long `.apply()` takes vs `pd.cut`, then compute the speedup.

> **About the sample:** the file is not shuffled, so the first 100k records happen to come from a
> single province. Use this sample to demonstrate *techniques*, not to produce population estimates.

In [ ]:
# Read only the first SAMPLE_N rows of the big Persons file, via an iterator.
with pd.read_stata(
    PERSONS_DTA,
    columns=['QID', 'PID', 'P04_AGE'],
    convert_categoricals=False,
    chunksize=SAMPLE_N,
) as it:
    persons = next(it).dropna(subset=['P04_AGE'])

print(f'Persons sampled (first {SAMPLE_N:,} rows): {len(persons):,}')

# TODO:
# 1. Define age_group(age) -> 'child' if age <= 14, 'working_age' if <= 64, else 'elderly'
# 2. Time: persons['age_apply'] = persons['P04_AGE'].apply(age_group)
# 3. Time: persons['age_cut'] = pd.cut(
#        persons['P04_AGE'], bins=[-1, 14, 64, 200],
#        labels=['child', 'working_age', 'elderly'])
# 4. Print both durations and the speedup factor
# 5. Compare value_counts() -- they should match

## B2. Conditional columns: `np.where` and `np.select`

Using the slim DataFrame from A2:
- `female_headed`: 1 if `DERH_HHSEX == 2` (Female), else 0  ->  use `np.where`
- `hh_size_class`: `"small"` (1-3), `"medium"` (4-6), `"large"` (7+)  ->  use `np.select`

In [ ]:
# Uses hh_slim from Part A2

# TODO:
# 1. hh_slim['female_headed'] = np.where(condition, value_if_true, value_if_false)
#    DERH_HHSEX == 2 means a female head of household
#
# 2. hh_slim['hh_size_class'] = np.select(
#        conditions=[...],   # list of boolean arrays
#        choicelist=[...],   # matching labels
#        default='unknown')
#    Bins: small = DERH_HSIZE <= 3, medium = <= 6, large = > 6
#
# 3. Print value_counts() for both new columns

## B3. Vectorised arithmetic with a guard

Build a per-household composition from the sampled Persons rows, then compute a
**child-to-adult ratio** — `n_children / n_adults`.

Some sampled households contain only children (no adults in the sample), so the denominator
can be zero. Guard it vectorised: `n_children / n_adults.where(n_adults > 0)`.

Then show how many households have no adult (undefined ratio) and how many have a ratio > 2.

In [ ]:
# Build per-household counts from the 100k-row Persons sample (children <= 14, adults >= 18)
comp = (
    persons
    .assign(
        is_child=(persons['P04_AGE'] <= 14).astype(int),
        is_adult=(persons['P04_AGE'] >= 18).astype(int),
    )
    .groupby('QID', as_index=False)
    .agg(
        n_members  = ('P04_AGE', 'size'),
        n_children = ('is_child', 'sum'),
        n_adults   = ('is_adult', 'sum'),
    )
)
print(f'Households in sample: {len(comp):,}')

# TODO:
# 1. comp['child_adult_ratio'] = n_children / n_adults.where(n_adults > 0)
# 2. Print: households with no adult (NaN ratio), households with ratio > 2
# 3. Show the distribution of child_adult_ratio with .describe()

---
# Part C — Chunked Reading (a backup technique)

> **Key constraint:** accumulate `(sum, count)` per group across chunks,
> then compute `mean = sum / count` *after* the loop. "Average of averages" is wrong
> unless every chunk has exactly the same size.

## C1. Chunked mean household size by population group

Compute mean `DERH_HSIZE` per `DERH_HHPOP` using the chunk accumulator pattern.
Do **not** store rows or average per-chunk means.

Reading with `convert_categoricals=False` keeps the codes numeric and avoids the
`CategoricalConversionWarning` that Stata raises when a labelled column is read in chunks.

In [ ]:
CHUNK_SIZE = 200_000   # ~7 chunks over 1.3M households

# TODO:
# 1. Open an iterator (use convert_categoricals=False to keep codes numeric):
#    pd.read_stata(HOUSEHOLDS_DTA, chunksize=CHUNK_SIZE,
#                  columns=['DERH_HHPOP', 'DERH_HSIZE'], convert_categoricals=False)
# 2. In each chunk:
#    a. drop rows where DERH_HHPOP or DERH_HSIZE is null
#    b. groupby DERH_HHPOP, accumulate sum and count of DERH_HSIZE
# 3. After the loop: mean = accumulated_sum / accumulated_count per group
# 4. Print how many chunks were processed
# 5. Display results sorted by mean descending (map codes via HHPOP_LABELS)

## C2. What cannot be done in chunks?

Some statistics require the full dataset before computing anything. Fill in the table.

| Statistic | Chunkable? | Reason |
|---|---|---|
| Count of households per province | | |
| Global median of `DERH_HSIZE` | | |
| Sum of `DERH_HSIZE` per district | | |
| 90th percentile of `P04_AGE` | | |
| Share of households with `DERH_HSIZE > 5` | | |

---
# Part D — File Formats: CSV, Excel, and Parquet

| Capability | CSV | Excel (.xlsx) | Parquet |
|---|---|---|---|
| Open with double-click | yes | yes | no |
| Preserves types | no | partial | yes |
| Compression | no | some | yes |
| Multiple sheets/tables | no | yes | no |
| Fast for large data | weak | no | yes |
| Read only some columns | no | weak | yes |
| Familiar to NSO staff | yes | yes | no |

## D1. Build the Parquet warehouse

Convert the Stata sources to Parquet **once**, then read them many times. We build two
analysis tables:

- `hh_main.parquet` — Households joined to Geography on `QID` (household grain, with Province)
- `persons_sample.parquet` — the **first `SAMPLE_N` rows** of the 347 MB Persons file

Because we read with `convert_categoricals=False`, every column is already a numeric code or a
plain string — there are no Stata categoricals to cast before writing Parquet.

In [ ]:
HH_COLS  = ['QID', 'DERH_HSIZE', 'DERH_HHPOP', 'H03_TENURE', 'DERH_HHSEX', 'DERH_HHAGE']
GEO_COLS = ['QID', 'Province', 'District', 'Geo_type']

# TODO:
# 1. Build hh_main:
#    - read Households (columns=HH_COLS, convert_categoricals=False)
#    - read Geography (columns=GEO_COLS, convert_categoricals=False)
#    - merge on 'QID' (how='left') and write to PARQUET_DIR / 'hh_main.parquet'
#
# 2. Build persons_sample:
#    - read the FIRST SAMPLE_N rows of Persons via a chunksize iterator (next(it))
#    - columns=['QID','PID','P02_SEX','P04_AGE','AGE_GROUP'], convert_categoricals=False
#    - write to PARQUET_DIR / 'persons_sample.parquet'
#
# 3. Print source size (MB), Parquet size (MB) and the compression ratio for hh_main.
#    Hint: Path.stat().st_size gives bytes

## D2. Column-selective timing: Parquet vs DTA

Time reading the five household columns from Parquet vs from the original Stata file.

In [ ]:
TASK_COLS = ['DERH_HSIZE', 'DERH_HHPOP', 'H03_TENURE', 'DERH_HHSEX', 'DERH_HHAGE']

# TODO:
# 1. Time pd.read_parquet(PARQUET_DIR / 'hh_main.parquet', columns=TASK_COLS)
# 2. Time pd.read_stata(HOUSEHOLDS_DTA, columns=TASK_COLS, convert_categoricals=False)
# 3. Print both durations and the speedup factor
# 4. Confirm both DataFrames have the same shape

## D3. Export a summary to CSV and Excel

Compute a province-level summary from `hh_main.parquet` and export it to both CSV and Excel.
These files are queried in Part E to show DuckDB working across formats.

In [ ]:
CSV_PATH = PARQUET_DIR / 'province_summary.csv'
XLS_PATH = PARQUET_DIR / 'province_summary.xlsx'

# TODO:
# 1. Load hh_main.parquet with columns=['Province','DERH_HSIZE','DERH_HHAGE']
# 2. groupby Province -> mean_hhsize, mean_headage, n_hh
# 3. Add a readable 'province_name' column with PROVINCE_LABELS
# 4. Export with .to_csv(CSV_PATH, index=False)
# 5. Export with .to_excel(XLS_PATH, index=False)
# 6. Print file sizes for both

## D4. Conversion recipes (reference — no TODO)

Keep these for future projects:

```python
# Excel -> Parquet
pd.read_excel('src.xlsx').to_parquet('dst.parquet')

# CSV -> Parquet (with explicit types)
pd.read_csv('src.csv', dtype={'province_code': 'str'}).to_parquet('dst.parquet')

# Parquet -> Excel (for stakeholders)
pd.read_parquet('data.parquet').to_excel('report.xlsx', index=False)

# Parquet -> CSV (for sharing)
pd.read_parquet('data.parquet').to_csv('export.csv', index=False)
```

---
# Part E — DuckDB: SQL Across Formats

| Situation | Tool |
|---|---|
| Aggregations over large files | DuckDB |
| Joining files of different formats | DuckDB |
| Reading only a few columns from a wide Parquet file | DuckDB |
| Statistical modelling | Pandas (after DuckDB preprocessing) |
| Quick in-memory DataFrame manipulation | Pandas |

In [ ]:
try:
    import duckdb
    print(f'duckdb {duckdb.__version__} ready')
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'duckdb', '-q'])
    import duckdb
    print(f'duckdb {duckdb.__version__} installed')

HH_PARQUET      = str(PARQUET_DIR / 'hh_main.parquet')
PERSONS_PARQUET = str(PARQUET_DIR / 'persons_sample.parquet')
CSV_PATH_STR    = str(CSV_PATH)
XLS_PATH_STR    = str(XLS_PATH)

## E1. Query Parquet — grouped aggregation

Replicate a grouped mean using a single SQL query on Parquet — no Python loop, no chunking.

In [ ]:
# TODO:
# Write a SQL query on read_parquet(HH_PARQUET) that:
#   - filters Province IS NOT NULL AND DERH_HSIZE IS NOT NULL
#   - groups by Province
#   - computes AVG(DERH_HSIZE) AS mean_hhsize, COUNT(*) AS n_hh
#   - orders by mean_hhsize DESC
# Run with duckdb.sql(...).to_df(); map Province -> PROVINCE_LABELS for readability

## E2. JOIN across Parquet files

Join `hh_main.parquet` (one row per household) to `persons_sample.parquet` (one row per person)
on `QID`, then compute **mean age (`P04_AGE`) by the household head's population group**
(`DERH_HHPOP`), with household and member counts.

> **Why group by population group, not province?** `persons_sample.parquet` is the first 100k
> records of the file, which all fall in one province — so province would give a single row.
> Population group varies within the sample, so it shows the join working. Treat this as a
> mechanics demo, not a population estimate.

> **Sanity check:** more members than households is expected (many members per household).

In [ ]:
# TODO:
# Write a SQL query joining the two Parquet files on QID:
#   FROM   read_parquet(HH_PARQUET)      AS hh
#   JOIN   read_parquet(PERSONS_PARQUET) AS p ON hh.QID = p.QID
#   WHERE  p.P04_AGE IS NOT NULL AND hh.DERH_HHPOP IS NOT NULL
#   GROUP  BY hh.DERH_HHPOP
#   SELECT hh.DERH_HHPOP,
#          AVG(p.P04_AGE)         AS mean_age,
#          COUNT(DISTINCT hh.QID) AS n_households,
#          COUNT(*)               AS n_members
#   ORDER  BY mean_age DESC
# Map DERH_HHPOP -> HHPOP_LABELS and sanity-check n_members >= n_households

## E3. Query CSV directly

DuckDB can query a CSV on disk without loading it into Python first. Query the province summary
CSV from D3 — no `pd.read_csv()` needed.

In [ ]:
# TODO:
# Use duckdb.sql(f"SELECT * FROM '{CSV_PATH_STR}' ORDER BY mean_hhsize DESC").to_df()
# Display all rows and print a note that no pd.read_csv() was needed.

## E4. Join formats: Parquet + Excel

DuckDB can join different formats in one query. Use the Excel file from D3 as a lookup table and
join it to the Parquet survey file.

**Approach:** read Excel with pandas -> register as a DuckDB view -> join in SQL.

In [ ]:
# TODO:
# 1. Load XLS_PATH with pd.read_excel() -> DataFrame called province_lookup
# 2. Register it: duckdb.register('province_lookup', province_lookup)
# 3. Write a SQL query:
#    FROM   read_parquet(HH_PARQUET) AS hh
#    JOIN   province_lookup          AS pl ON hh.Province = pl.Province
#    GROUP  BY hh.Province, pl.mean_hhsize
#    SELECT hh.Province,
#           pl.mean_hhsize      AS benchmark_hhsize,
#           AVG(hh.DERH_HSIZE)  AS actual_hhsize,
#           COUNT(*)            AS n_hh
#    ORDER  BY hh.Province
# The two hhsize columns should be identical (same underlying data) -- confirms the join works

## E5. Persist a result as Parquet with `COPY TO`

DuckDB can write query results straight to Parquet, bypassing Python memory.

```sql
COPY (...query...) TO 'output.parquet' (FORMAT PARQUET);
```

Write the population-group age summary from E2 to
`PARQUET_DIR / 'group_age_summary.parquet'`, then read it back with pandas to verify.

In [ ]:
OUT_PATH = str(PARQUET_DIR / 'group_age_summary.parquet')

# TODO:
# 1. duckdb.sql(f"COPY (...your E2 query...) TO '{OUT_PATH}' (FORMAT PARQUET)")
# 2. Read back with pd.read_parquet(OUT_PATH)
# 3. Print shape and display the rows

---
# Summary — When to use each technique

Fill in the table based on what you observed in this notebook.

| Situation | Recommended approach |
|---|---|
| Wide file, need 5 of 32 columns | |
| A 347 MB file on a shared server | |
| Keeping survey codes numeric | |
| Classify ages into groups | |
| Same file queried many times | |
| File larger than RAM, need a grouped mean | |
| JOIN two large files without loading either | |
| Query a CSV without loading it | |
| Share results with a non-Python colleague | |